In [0]:

from pyspark.sql import functions as F

# Read published tables independently
bronze = spark.table("workspace.bronze.yellow_taxi_trips")
silver = spark.table("workspace.silver.yellow_taxi_trips")
quarantine = spark.table("workspace.silver.yellow_taxi_quarantine")
outside = spark.table("workspace.silver.yellow_taxi_out_of_scope")
fact = spark.table("workspace.gold.fact_yellow_trip")
daily = spark.table("workspace.gold.agg_daily_pickup_zone")
monthly = spark.table("workspace.gold.agg_monthly_kpi")

# Check source-to-Gold record reconciliation
bronze_count = bronze.count()
silver_count = silver.count()
quarantine_count = quarantine.count()
outside_count = outside.count()
fact_count = fact.count()

assert bronze_count == 3475226
assert bronze_count == silver_count + quarantine_count
assert silver_count == fact_count + outside_count

print("PASS: Source-to-Gold record reconciliation")

# Independently compare financial totals
fact_totals = fact.agg(
    F.count("*").alias("trips"),
    F.sum(F.col("total_amount").cast("decimal(18,2)")).alias("charges"),
    F.sum(F.col("fare_amount").cast("decimal(18,2)")).alias("fares")
).first()

daily_totals = daily.agg(
    F.sum("accepted_trip_count").alias("trips"),
    F.sum("total_recorded_charges").alias("charges"),
    F.sum("total_fares").alias("fares")
).first()

assert fact_totals.asDict() == daily_totals.asDict()

monthly_row = monthly.first()
assert monthly.count() == 1
assert monthly_row["source_month"] == "2025-01"
assert monthly_row["accepted_trip_count"] == fact_totals["trips"]
assert monthly_row["total_recorded_charges"] == fact_totals["charges"]
assert monthly_row["total_fares"] == fact_totals["fares"]

print("PASS: Fact, daily and monthly financial reconciliation")
print("Accepted trips:", fact_count)
print("Recorded charges:", fact_totals["charges"])
print("Recorded fares:", fact_totals["fares"])
print("ALL VALIDATION CHECKS PASSED")


 Inspect Delta table history

In [0]:

# Inspect recent Bronze Delta table operations
history = spark.sql("""
    DESCRIBE HISTORY workspace.bronze.yellow_taxi_trips
""")

display(
    history.select(
        "version",
        "timestamp",
        "operation"
    ).orderBy("version", ascending=False).limit(5)
)


In [0]:

# Inspect recent Bronze table versions
history = spark.sql("""
    DESCRIBE HISTORY workspace.bronze.yellow_taxi_trips
""")

display(
    history.select(
        "version",
        "timestamp",
        "operation"
    )
    .orderBy("version", ascending=False)
    .limit(10)
)
